In [ ]:
# Block 0 — setup. Code from GitHub onto the VM disk; data + embeddings on Drive.
import os, sys
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = "/content/NeoAntigen"
try:   # private repo: store a GitHub token as Colab secret GH_TOKEN
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = ""
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/Tienanh209/NeoAntigen.git"
if not os.path.exists(REPO_DIR):
    !git clone {URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

# numpy/scipy/torch are filtered out: Colab's torch is built against its own numpy
!grep -viE '^(torch|torchvision|torchaudio|nvidia-|numpy|scipy|#)' requirements.txt > req_colab.txt
!pip install -q -r req_colab.txt

In [ ]:
# Block 1 — dataset, embedding caches, shared settings.
# Every experiment block below depends on THIS cell and on nothing else:
# they communicate only through the results store on Drive.
import pandas as pd
from src import config, data, embed, results, runner

df = data.load_dataset()
BID = data.build_id(df)
MAXLEN = data.max_len(df)
SEEDS = (42, 43, 44)        # 3 seeds: at ~127 positives the seed spread rivals the effects compared
PLM = "prott5"              # feature set for the hyperparameter sweeps, as in the paper
FORCE = False               # True re-runs committed configurations

for plm in ("prott5", "esm2_650m", "esm1b"):
    if embed.is_cached(plm, BID):
        embed.stage(plm, BID)
    else:
        print(f"!! {plm} not cached — run embeddings.ipynb first")

print("build id:", BID, "| max_len:", MAXLEN, "| results:", config.RESULTS_DIR)
print(df.groupby(["split", "label"]).size().unstack(fill_value=0))

In [ ]:
# Block 2 — Experiment A: single window sizes (paper Table 2)
cfgs = [dict(label=f"window {w}", features={"kind": "plm", "plm": PLM},
             model={"kind": "mcnn", "windows": [w], "filters": 512})
        for w in (8, 16, 24, 32, 34)]
runner.run_series("A_window", cfgs, df=df, seeds=SEEDS, force=FORCE)

print("\nIndependent test")
display(results.fmt(results.table("A_window")))
print("5-fold cross-validation")
display(results.fmt(results.table("A_window", eval_split="cv")))
results.roc("A_window", title="Single window sizes — independent test");

In [ ]:
# Block 3 — Experiment B: window combinations (paper Table 3)
combos = [[34], [34, 32], [34, 32, 24], [34, 32, 24, 16], [34, 32, 24, 16, 8]]
cfgs = [dict(label=str(c), features={"kind": "plm", "plm": PLM},
             model={"kind": "mcnn", "windows": c, "filters": 512}) for c in combos]
runner.run_series("B_combo", cfgs, df=df, seeds=SEEDS, force=FORCE)

display(results.fmt(results.table("B_combo")))
display(results.fmt(results.table("B_combo", eval_split="cv")))
results.roc("B_combo", title="Window combinations — independent test");

In [ ]:
# Block 4 — Experiment C: filters per window (paper Table 4)
# conv_params is in the table on purpose: at 2048 filters the convolutions hold
# ~300M parameters for ~1.1k training peptides, which is the point of this sweep.
WINDOWS = [34, 32, 24, 16]
cfgs = [dict(label=f"{f} filters", features={"kind": "plm", "plm": PLM},
             model={"kind": "mcnn", "windows": WINDOWS, "filters": f,
                    "dropout": 0.5 if f >= 512 else 0.3},
             train_cfg={"batch_size": 16} if f >= 1024 else None)
        for f in (128, 256, 512, 1024, 2048)]
runner.run_series("C_filters", cfgs, df=df, seeds=SEEDS, force=FORCE)

display(results.fmt(results.table("C_filters", extra_cols=("conv_params",))))
results.roc("C_filters", title="Filters per window — independent test");

In [ ]:
# Block 5 — Experiment D: pre-trained language model features (paper Table 5, Fig 4a)
BEST_WINDOWS, BEST_FILTERS = [34, 32, 24, 16], 512
cfgs = [dict(label=name, features={"kind": "plm", "plm": plm},
             model={"kind": "mcnn", "windows": BEST_WINDOWS, "filters": BEST_FILTERS})
        for plm, name in [("prott5", "ProtTrans"), ("esm2_650m", "ESM-2"), ("esm1b", "ESM-1b")]]
runner.run_series("D_plm", cfgs, df=df, seeds=SEEDS, force=FORCE)

print("Cross-validation (5 folds)")
display(results.fmt(results.table("D_plm", eval_split="cv")))
print("Independent test")
display(results.fmt(results.table("D_plm")))
results.roc("D_plm", title="PLM features — independent test");

In [ ]:
# Block 6 — Experiment E: classifiers (paper Table 6, Fig 4b)
# KNN / RF / SVM take the mean-pooled embedding; the mCNN keeps per-residue input.
cfgs = [dict(label=n.upper(), features={"kind": "plm_pooled", "plm": PLM},
             model={"kind": "sklearn", "name": n}) for n in ("knn", "rf", "svm")]
cfgs.append(dict(label="mCNN", features={"kind": "plm", "plm": PLM},
                 model={"kind": "mcnn", "windows": [34, 32, 24, 16], "filters": 512}))
runner.run_series("E_clf", cfgs, df=df, seeds=SEEDS, force=FORCE)

display(results.fmt(results.table("E_clf")))
results.roc("E_clf", title="Classifiers — independent test");

In [ ]:
# Block 7 — Experiment F: MHC-I vs MHC-II (paper Table 7)
# No training: this re-scores the best stored run on two subsets of its own predictions.
rid = results.best_run("D_plm")
test = data.add_mhc_class(df[df["split"] == "test"])
subsets = {"Independent test": test["uid"],
           "MHC-I": test.loc[test["mhc"] == "I", "uid"],
           "MHC-II": test.loc[test["mhc"] == "II", "uid"]}
print("best run on CV:", rid)
display(results.fmt(results.breakdown(rid, subsets)))

In [ ]:
# Block 8 — Experiment G: the baseline ladder (CS535 E0-E3)
# How much of the performance comes from composition alone, from order, from
# pretraining, and from local scanning on top of pretraining.
cfgs = [
    dict(label="composition + LogReg", features={"kind": "aac"},
         model={"kind": "sklearn", "name": "logreg"}),
    dict(label="one-hot CNN (w=8)", features={"kind": "onehot"},
         model={"kind": "mcnn", "windows": [8], "filters": 256}),
    dict(label="one-hot mCNN", features={"kind": "onehot"},
         model={"kind": "mcnn", "windows": [34, 32, 24, 16], "filters": 256}),
    dict(label="pooled PLM + MLP", features={"kind": "plm_pooled", "plm": PLM},
         model={"kind": "mlp", "hidden": [256]}),
    dict(label="PLM + mCNN", features={"kind": "plm", "plm": PLM},
         model={"kind": "mcnn", "windows": [34, 32, 24, 16], "filters": 512}),
]
runner.run_series("G_ladder", cfgs, df=df, seeds=SEEDS, force=FORCE)

display(results.fmt(results.table("G_ladder")))
results.roc("G_ladder", title="Baseline ladder — independent test");

In [ ]:
# Block 9 — Experiment H: class weighting (CS535 E7)
# At ~1 positive per 13 negatives, this is what moves sensitivity against precision.
base = dict(features={"kind": "plm", "plm": PLM},
            model={"kind": "mcnn", "windows": [34, 32, 24, 16], "filters": 512})
cfgs = [dict(label="weighted BCE", train_cfg={"class_weight": True}, **base),
        dict(label="unweighted BCE", train_cfg={"class_weight": False}, **base)]
runner.run_series("H_weight", cfgs, df=df, seeds=SEEDS, force=FORCE)

print("Tuned threshold (chosen on CV)")
display(results.fmt(results.table("H_weight")))
print("Fixed 0.5 threshold — note how differently 0.5 behaves between the two")
display(results.fmt(results.table("H_weight", thr_mode="fixed")))

In [ ]:
# Block 10 — Experiment I: similarity leakage (CS535 E8)
# Random folds let near-duplicate peptides sit on both sides of the split;
# grouped folds keep peptides that share an 8-mer together.
base = dict(features={"kind": "plm", "plm": PLM},
            model={"kind": "mcnn", "windows": [34, 32, 24, 16], "filters": 512})
cfgs = [dict(label="random folds", split_mode="stratified", **base),
        dict(label="similarity-controlled folds", split_mode="grouped", **base)]
runner.run_series("I_leak", cfgs, df=df, seeds=SEEDS, force=FORCE)

print("Cross-validation — the leakage shows up here, not on the test split")
display(results.fmt(results.table("I_leak", eval_split="cv")))
display(results.fmt(results.table("I_leak")))

g = data.kmer_groups(df)
print(f"\n{len(df)} peptides fall into {g.nunique()} shared-8-mer groups; "
      f"largest group has {g.value_counts().iloc[0]}")

In [ ]:
# Block 11 — report: every table and figure, rebuilt from the store. No training.
# This is what to run after a disconnect to see the whole state of the project.
TITLES = {"A_window": "Single window sizes", "B_combo": "Window combinations",
          "C_filters": "Filters per window", "D_plm": "PLM features",
          "E_clf": "Classifiers", "G_ladder": "Baseline ladder",
          "H_weight": "Class weighting", "I_leak": "Similarity leakage"}

for exp_id, title in TITLES.items():
    t = results.table(exp_id)
    if t.empty:
        print(f"--- {title}: not run yet\n")
        continue
    print(f"--- {title} ({exp_id}) — independent test")
    display(results.fmt(t))
    results.roc(exp_id, title=f"{title} — independent test")

runs = results.load_runs()
print(f"\n{len(runs)} committed runs in {config.RESULTS_DIR}")